In [2]:
print("Hello !")


Hello !


In [7]:
import pandas as pd
from pathlib import Path

data_folder = Path("archive")

train = pd.read_csv(data_folder / "UNSW_NB15_training-set.csv")
test = pd.read_csv(data_folder / "UNSW_NB15_testing-set.csv")

print("Training dataset:", train.shape)
print("Testing dataset:", test.shape)

display(train.head())


Training dataset: (82332, 45)
Testing dataset: (175341, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


In [8]:
print("Column names:")
print(train.columns.tolist())

print("\nMissing values:")
print(train.isnull().sum().sum())

print("\nTarget distribution:")
print(train["label"].value_counts())

print("\nAttack categories:")
print(train["attack_cat"].value_counts())

Column names:
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

Missing values:
0

Target distribution:
label
1    45332
0    37000
Name: count, dtype: int64

Attack categories:
attack_cat
Normal            37000
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44
Name: count, dtype: int64


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Features (inputs) and target (answer)
X_train = train.drop(columns=["label", "attack_cat", "id"])
y_train = train["label"]

X_test = test.drop(columns=["label", "attack_cat", "id"])
y_test = test["label"]

# Identify categorical and numerical columns
categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

# Convert text categories into numeric features
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"),
         categorical_cols),
        ("numerical", "passthrough", numerical_cols)
    ]
)

# Create the model pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

# Train the model
model.fit(X_train, y_train)

print("Model training completed!")

C:\Users\babak\AppData\Local\Temp\ipykernel_21760\1865197015.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


Model training completed!


In [10]:
# Predict on unseen test data
y_pred = model.predict(X_test)

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Normal", "Attack"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

      Normal       0.78      0.98      0.86     56000
      Attack       0.99      0.87      0.92    119341

    accuracy                           0.90    175341
   macro avg       0.88      0.92      0.89    175341
weighted avg       0.92      0.90      0.90    175341

Confusion Matrix:
[[ 54720   1280]
 [ 15809 103532]]


In [11]:
import json
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "attack_precision": precision_score(y_test, y_pred),
    "attack_recall": recall_score(y_test, y_pred),
    "attack_f1": f1_score(y_test, y_pred)
}

Path("reports").mkdir(exist_ok=True)

with open("reports/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print(json.dumps(metrics, indent=4))

{
    "accuracy": 0.9025384821576243,
    "attack_precision": 0.9877876579017669,
    "attack_recall": 0.8675308569561173,
    "attack_f1": 0.923761894777228
}


In [12]:
import json
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "attack_precision": precision_score(y_test, y_pred),
    "attack_recall": recall_score(y_test, y_pred),
    "attack_f1": f1_score(y_test, y_pred)
}

Path("reports").mkdir(exist_ok=True)

with open("reports/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print(json.dumps(metrics, indent=4))

{
    "accuracy": 0.9025384821576243,
    "attack_precision": 0.9877876579017669,
    "attack_recall": 0.8675308569561173,
    "attack_f1": 0.923761894777228
}


In [13]:
import json
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "attack_precision": precision_score(y_test, y_pred),
    "attack_recall": recall_score(y_test, y_pred),
    "attack_f1": f1_score(y_test, y_pred)
}

Path("reports").mkdir(exist_ok=True)

with open("reports/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print(json.dumps(metrics, indent=4))

{
    "accuracy": 0.9025384821576243,
    "attack_precision": 0.9877876579017669,
    "attack_recall": 0.8675308569561173,
    "attack_f1": 0.923761894777228
}


In [14]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)

print("LOGISTIC REGRESSION RESULTS")
print(classification_report(
    y_test,
    logistic_pred,
    target_names=["Normal", "Attack"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, logistic_pred))


c:\Users\babak\OneDrive\Υπολογιστής\machine learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LOGISTIC REGRESSION RESULTS
              precision    recall  f1-score   support

      Normal       0.56      0.94      0.70     56000
      Attack       0.96      0.65      0.77    119341

    accuracy                           0.74    175341
   macro avg       0.76      0.79      0.74    175341
weighted avg       0.83      0.74      0.75    175341

Confusion Matrix:
[[52638  3362]
 [41786 77555]]


In [15]:

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix

scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("numerical", StandardScaler(), numerical_cols)
    ]
)

scaled_logistic_model = Pipeline([
    ("preprocessor", scaled_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

scaled_logistic_model.fit(X_train, y_train)

scaled_logistic_pred = scaled_logistic_model.predict(X_test)

print("SCALED LOGISTIC REGRESSION RESULTS")
print(classification_report(
    y_test,
    scaled_logistic_pred,
    target_names=["Normal", "Attack"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, scaled_logistic_pred))


SCALED LOGISTIC REGRESSION RESULTS
              precision    recall  f1-score   support

      Normal       0.71      0.97      0.82     56000
      Attack       0.98      0.81      0.89    119341

    accuracy                           0.86    175341
   macro avg       0.85      0.89      0.86    175341
weighted avg       0.90      0.86      0.87    175341

Confusion Matrix:
[[54440  1560]
 [22328 97013]]


In [16]:

import joblib
from pathlib import Path

Path("models").mkdir(exist_ok=True)

joblib.dump(model, "models/random_forest_model.joblib")

print("Random Forest model saved successfully!")
print("Location: models/random_forest_model.joblib")



Random Forest model saved successfully!
Location: models/random_forest_model.joblib


: 